In [3]:
import os
import glob

import json
import numpy as np
import pandas as pd

from pathlib import Path
from typing import List, Tuple, Any, Optional
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor

from concurrent.futures import ProcessPoolExecutor, as_completed

In [4]:
# STORAGE_DIR = "Z:\\starcraft-vision\\"
STORAGE_DIR = "/workspace/"

DATA_DIR = os.path.join(STORAGE_DIR, "data")
GROUND_TRUTH_DIR = os.path.join(DATA_DIR, "label", "dst")
PREDICTIONS_DIR = os.path.join(STORAGE_DIR, "predictions")

In [5]:
REPLAYS = [1725]
# REPLAYS = [36, 212, 438, 522, 1660, 6254]
# REPLAYS = [1559, 1628, 2351, 6219, 11251]
# REPLAYS = [275, 1725, 3613, 4520, 4664]
# REPLAYS = [36, 212, 438, 522, 1660, 6254, 1559, 1628, 2351, 6219, 11251, 275, 1725, 3613, 4520, 4664]

REPLAYS = [str(_) + '.rep' for _ in REPLAYS]

In [6]:
# TARGET_MODEL = "test_input_model/model_020"
# TARGET_MODEL = "all_correct_win4_b16_20250826_045543/model_030"
# TARGET_MODEL = "all_correct_win4_b16_kbrs_20250821_044349/model_015"

In [7]:
ground_truth_json_path = [
    f for name in REPLAYS
    for f in glob.glob(os.path.join(GROUND_TRUTH_DIR, name, '*.json'))
]

ground_truth_json_path

['/workspace/data/label/dst/1725.rep/all_correct.json']

In [8]:
def _build_json_paths(directory: str,
                      label_method: str = "all_correct",
                      replays: Optional[List[str]] = None) -> List[str]:
    directory = os.path.abspath(directory)
    if replays:
        return [os.path.join(directory, rep, f"{label_method}.json") for rep in replays]
    return [str(p) for p in Path(directory).glob("**/*.json")]

def _read_one_json(path: str, use_orjson: bool = True) -> Tuple[str, Any, Optional[str]]:
    try:
        if use_orjson:
            try:
                import orjson
                with open(path, "rb") as f:
                    return path, orjson.loads(f.read()), None
            except ModuleNotFoundError:
                pass  # orjson이 없으면 표준 json로 폴백
        with open(path, "r", encoding="utf-8") as f:
            return path, json.load(f), None
    except Exception as e:
        return path, None, f"{type(e).__name__}: {e}"

def read_json_files_threaded(directory: str,
                             label_method: str = "all_correct",
                             replays: Optional[List[str]] = None,
                             max_workers: Optional[int] = None) -> List[Any]:
    paths = _build_json_paths(directory, label_method, replays)
    if not paths:
        return []

    if max_workers is None:
        cpu = os.cpu_count() or 4
        max_workers = min(32, cpu * 5)

    results: List[Any] = []
    errors: List[str] = []

    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        for path, data, err in ex.map(_read_one_json, paths):
            if err:
                errors.append(f"[FAIL] {path} -> {err}")
            else:
                results.append()

    if errors:
        print("\n".join(errors))
    return results

def read_json_files_processes(directory: str,
                              label_method: str = "all_correct",
                              replays: Optional[List[str]] = None,
                              max_workers: Optional[int] = None) -> List[Any]:
    paths = _build_json_paths(directory, label_method, replays)
    if not paths:
        return []

    if max_workers is None:
        max_workers = os.cpu_count() or 4

    results: List[Any] = []
    errors: List[str] = []

    with ProcessPoolExecutor(max_workers=max_workers) as ex:
        for path, data, err in ex.map(_read_one_json, paths):
            if err:
                errors.append(f"[FAIL] {path} -> {err}")
            else:
                results.append(data)

    if errors:
        print("\n".join(errors))
    return results

In [9]:
ground_truth_data = read_json_files_processes(GROUND_TRUTH_DIR, label_method='all_correct', replays=REPLAYS, max_workers=8)

In [10]:
def get_annotations(data, replay_suffix=".rep"):
    frames = []

    for item in (data or []):
        anns = item.get("annotations", [])
        if not anns:
            continue

        df = pd.DataFrame(anns)

        desc = (item.get("info") or {}).get("description", "") or ""
        replay_id = None
        if isinstance(desc, str) and desc.strip():
            replay_id = desc.strip().split()[-1]

        if replay_id is None:
            replay_id = "unknown"

        if replay_suffix and not str(replay_id).endswith(replay_suffix):
            replay_id = f"{replay_id}{replay_suffix}"

        df["replay"] = replay_id
        frames.append(df)

    base_cols = ["replay", "id", "image_id", "category_id", "bbox", "area", "segmentation", "iscrowd"]
    if not frames:
        return pd.DataFrame(columns=base_cols + ["score"])

    has_score = any("score" in f.columns for f in frames)

    wanted_cols = base_cols + (["score"] if has_score else [])
    for i, f in enumerate(frames):
        for col in wanted_cols:
            if col not in f.columns:
                frames[i][col] = pd.NA

    annotations = pd.concat(frames, ignore_index=True)

    return annotations[wanted_cols]

In [11]:
annotation_ground_truth = get_annotations(ground_truth_data)
# annotation_prediction = get_annotations(prediction_data)

In [12]:
# def dataframe_topk_score(dataframe, K=1):
#     topk_df = (
#       dataframe.sort_values('score', ascending=False)
#         .groupby(['replay', 'image_id'], group_keys=False)
#         .head(K)
#     )
#     return topk_df.sort_values(['replay', 'image_id'])

In [13]:
# top1_annotation_prediction = dataframe_topk_score(annotation_prediction, K=1)

In [14]:
annotation_ground_truth["hid"] = annotation_ground_truth.groupby(["replay", "image_id"]).cumcount()  # 0~4 인덱스 생성
wide_ground_truth = annotation_ground_truth.pivot(index=["replay", "image_id"], columns="hid", values="bbox")
wide_ground_truth.columns = [f"gt_{i}" for i in wide_ground_truth.columns]
wide_ground_truth.reset_index(inplace=True)


In [15]:
item = wide_ground_truth.iloc[17500]

In [16]:
item_npy = os.path.join(DATA_DIR, 'input', 'dst', item['replay'], str(item['image_id']) + '.npy')

In [17]:
def get_item_obj(item):
    obj = np.load(item)
    return obj

In [18]:
from matplotlib import pyplot as plt

mkdir -p failed for path /.config/matplotlib: [Errno 13] Permission denied: '/.config'
Matplotlib created a temporary cache directory at /tmp/matplotlib-e6dnsu3t because there was an issue with the default path (/.config/matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


In [19]:
from enum import Enum

class Channel(Enum):
    Player_1_Worker = 0
    Player_1_Ground = 1
    Player_1_Air = 2
    Player_1_Building = 3
    
    Player_2_Worker = 4
    Player_2_Ground = 5
    Player_2_Air = 6
    Player_2_Building = 7
    
    Resource = 8
    Vision = 9
    Terrain = 10

In [20]:
item_obj = np.load(item_npy)

In [21]:
kw, kh = 20, 12
stride = 1

def score_density(x: np.ndarray) -> np.ndarray:
    _, H, W = x.shape
    
    x_sum = x.sum(axis=0, keepdims=True).astype(np.float32, copy=False)  # (N,H,W)
    x_sum = x_sum.reshape(H, W).astype(np.float32, copy=False)  # (H,W)

    ii = np.pad(x_sum, ((1,0),(1,0)), mode='constant')
    ii = ii.cumsum(axis=0).cumsum(axis=1)
    
    win_sum = (
        ii[kh:, kw:]    # bottom-right
        - ii[:-kh, kw:] # top-right
        - ii[kh:, :-kw] # bottom-left
        + ii[:-kh, :-kw]# top-left
    )
    
    den = win_sum / float(kh * kw)                       # (oh,ow)
    # return den
    return (den - den.min()) / (den.max() - den.min())

In [22]:
from numpy.lib.stride_tricks import sliding_window_view

def gaussian_kernel2d(kw, kh, sigma=None, dtype=np.float32):
    if sigma is None:
        sigma_y = max(kh, 1) / 3.0
        sigma_x = max(kw, 1) / 3.0
    elif np.isscalar(sigma):
        sigma_y = sigma_x = float(sigma)
    else:
        sigma_y, sigma_x = map(float, sigma)

    y = np.arange(kh, dtype=dtype) - (kh - 1) / 2.0
    x = np.arange(kw, dtype=dtype) - (kw - 1) / 2.0
    Y, X = np.meshgrid(y, x, indexing='ij')
    K = np.exp(-(Y**2)/(2*sigma_y**2) - (X**2)/(2*sigma_x**2)).astype(dtype, copy=False)
    K /= (K.sum() + 1e-6)
    return K  # (kh, kw), sum≈1


def score_centeredness(x: np.ndarray, stride=1, sigma=None) -> np.ndarray:
    _, H, W = x.shape
    
    x_sum = x.sum(axis=0, keepdims=True).astype(np.float32, copy=False)
    x_sum = x_sum.reshape(H, W).astype(np.float32, copy=False)  # (H,W)
    
    K = gaussian_kernel2d(kw, kh, sigma=sigma, dtype=np.float32)  # (kh, kw), sum=1
    denom = float(K.sum()) + 1e-6
    
    patches = sliding_window_view(x_sum, (kh, kw))                     # (N, H-kh+1, W-kw+1, kh, kw)

    cen = (patches * K).sum(axis=(-1, -2))                                  # (N, H-kh+1, W-kw+1)
    
    # 평균 정규화 (PyTorch 경로와 동일)
    denom = float(K.sum()) + 1e-6
    cen = cen / denom

    # 입력 dtype 정책: 부동소수면 입력 dtype으로 반환, 정수 입력이면 float32 유지
    if x.dtype.kind == 'f':
        cen = cen.astype(x.dtype, copy=False)
    # return cen
    return (cen - cen.min()) / (cen.max() - cen.min())

In [23]:
def get_projection(x: np.ndarray, axis: int, keepdims: bool = False) -> np.ndarray:
    """입력 배열 x에 대해 지정된 축(axis)을 따라 1차원 투영을 수행합니다.
    
    Args:
        x (np.ndarray): 입력 배열.
        axis (int): 투영을 수행할 축. 0은 높이, 1은 너비를 나타냅니다.
        keepdims (bool): True로 설정하면 투영 후에도 차원 수를 유지합니다. 기본값은 False입니다.
    
    Returns:
        np.ndarray: 투영된 배열.
    """
    if axis not in (0, 1):
        raise ValueError("axis는 0(높이) 또는 1(너비)이어야 합니다.")
    
    # 축을 따라 합산하여 투영 수행
    projection = x.sum(axis=0 if axis == 0 else 1, keepdims=keepdims)
    
    return projection

In [24]:
def score_mixture(x: np.ndarray, stride=1, mode='entropy') -> np.ndarray:
    _, H, W = x.shape
    eps = 1e-6
    pow_k = 1.0  # 0.5, 1.0, 2.0
    
    # ONES = np.ones((kh, kw), dtype=np.float32)
    xA = x[0:4,...].sum(axis=0, keepdims=False).astype(np.float32, copy=False)  # (H,W)
    xB = x[4:8,...].sum(axis=0, keepdims=False).astype(np.float32, copy=False)  # (H,W)
    
    def _sum_pool2d(x1):
        ii = np.pad(x1, ((1,0),(1,0)), mode='constant')
        ii = ii.cumsum(axis=0).cumsum(axis=1)
        
        win_sum = (
            ii[kh:, kw:]    # bottom-right
            - ii[:-kh, kw:] # top-right
            - ii[kh:, :-kw] # bottom-left
            + ii[:-kh, :-kw]# top-left
        )
        return win_sum
    
    A = _sum_pool2d(xA)  # (oh,ow)
    B = _sum_pool2d(xB)  # (oh,ow)
    
    den = np.maximum(A + B, eps, dtype=np.float32)                       # (oh,ow)
    p = A / den  # (oh,ow)
    
    if mode == 'entropy':
        p1 = np.clip(p, eps, 1.0 - eps)
        q1 = np.clip(1.0 - p, eps, 1.0 - eps)
        
        mix = -(p1 * np.log(p1) + q1 * np.log(q1)) / np.log(2.0)  # (oh,ow), [0,1]
    elif mode == 'gini':
        mix = 2.0 * p * (1.0 - p)  # (oh,ow), [0,0.5]
    elif mode == 'linear':
        mix = 1.0 - np.abs(2.0 * p - 1.0)  # (oh,ow), [0,1]
    else:
        mix = 4.0 * p * (1.0 - p)  # (oh,ow), [0,1]
        
    if pow_k != 1.0:
        mix = np.clip(mix, 0.0, 1.0) ** pow_k  # (oh,ow), [0,1]
        
    # return mix, A, B
    return (mix - mix.min()) / (mix.max() - mix.min())

In [25]:
def overview(ax, x: np.ndarray):
    # Terrain & resource
    ax.imshow(x[Channel.Terrain.value] > 0, cmap='Greys', alpha=0.5)
    resource_alpha = np.where(x[Channel.Resource.value] == 1, 1.0, 0.0)
    ax.imshow(x[Channel.Resource.value] == 1, cmap='BuGn', alpha=resource_alpha)
    
    # Player colors
    p1_cmap = 'Greens'
    p2_cmap = 'Reds'
    
    # Player 1 & 2 - worker
    player_1_worker_alpha = np.where(x[Channel.Player_1_Worker.value] != 0, 1.0, 0.0)
    player_2_worker_alpha = np.where(x[Channel.Player_2_Worker.value] != 0, 1.0, 0.0)
    ax.imshow(x[Channel.Player_1_Worker.value] != 0, cmap=p1_cmap, alpha=player_1_worker_alpha)
    ax.imshow(x[Channel.Player_2_Worker.value] != 0, cmap=p2_cmap, alpha=player_2_worker_alpha)
    
    # Player 1 & 2 - ground
    player_1_ground_alpha = np.where(x[Channel.Player_1_Ground.value] != 0, 1.0, 0.0)
    player_2_ground_alpha = np.where(x[Channel.Player_2_Ground.value] != 0, 1.0, 0.0)
    ax.imshow(x[Channel.Player_1_Ground.value] != 0, cmap=p1_cmap, alpha=player_1_ground_alpha)
    ax.imshow(x[Channel.Player_2_Ground.value] != 0, cmap=p2_cmap, alpha=player_2_ground_alpha)
    
    # Player 1 & 2 - air
    player_1_air_alpha = np.where(x[Channel.Player_1_Air.value] != 0, 1.0, 0.0)
    player_2_air_alpha = np.where(x[Channel.Player_2_Air.value] != 0, 1.0, 0.0)
    ax.imshow(x[Channel.Player_1_Air.value] != 0, cmap=p1_cmap, alpha=player_1_air_alpha)
    ax.imshow(x[Channel.Player_2_Air.value] != 0, cmap=p2_cmap, alpha=player_2_air_alpha)

    # Player 1 & 2 - building
    player_1_building_alpha = np.where(x[Channel.Player_1_Building.value] != 0, 1.0, 0.0)
    player_2_building_alpha = np.where(x[Channel.Player_2_Building.value] != 0, 1.0, 0.0)
    ax.imshow(x[Channel.Player_1_Building.value] != 0, cmap=p1_cmap, alpha=player_1_building_alpha)
    ax.imshow(x[Channel.Player_2_Building.value] != 0, cmap=p2_cmap, alpha=player_2_building_alpha)

    # Vision mask
    vision_alpha = np.where(x[Channel.Vision.value] == 1, 0.0, 0.9)
    ax.imshow(np.zeros_like(x[Channel.Vision.value]), cmap='Greys_r', alpha=vision_alpha)

In [26]:
from matplotlib.animation import FuncAnimation, PillowWriter # GIF용, mp4는 FFMpegWriter

In [27]:
GT_COLS = [f'gt_{i}' for i in range(5)]
GRID_W = GRID_H = 128  # 좌표계

In [28]:
def _extract_gts(item_row):
    gts = []
    for k in GT_COLS:
        if k in item_row:
            v = item_row[k]
            if v is None: 
                continue
            arr = np.asarray(v, dtype=float).ravel()
            if arr.size == 4 and not np.any(np.isnan(arr)):
                x, y, w, h = arr
                gts.append((x,y,w,h))
    return gts


In [29]:
def _overview_init(ax, arr):
    before = set(ax.get_children())
    overview(ax, arr)  # 기존 함수 호출 (ax에 아티스트 생성)
    new = [a for a in ax.get_children() if a not in before]
    for a in new:
        try: a.set_animated(True)
        except: pass
    return new  # blit 대상 목록

In [30]:
def _overview_update(ax, arr, dyn_list):
    for a in dyn_list[:]:
        try: a.remove()
        except: pass
    dyn_list.clear()
    before = set(ax.get_children())
    overview(ax, arr)
    new = [a for a in ax.get_children() if a not in before]
    for a in new:
        try: a.set_animated(True)
        except: pass
    dyn_list.extend(new)
    return dyn_list

In [ ]:
# ---- 워커: 스코어 사전계산 (메모리맵 입력) ----
def _worker_compute(data_dir, replay, image_id, weights):
    path = os.path.join(data_dir, "input", "dst", replay, f"{image_id}.npy")
    arr  = np.load(path, mmap_mode="r")
    den  = score_density(arr).astype(np.float32, copy=False)        # [0,1] 가정
    cen  = score_centeredness(arr).astype(np.float32, copy=False)   # [0,1] 가정
    mix  = score_mixture(arr, mode="linear").astype(np.float32, copy=False)  # [0,1] 가정

    # BRS를 워커에서 계산(메인 스레드 부담↓). 단순 가중 평균 후 [0,1] 클립.
    sw   = weights['density'] + weights['mixture'] + weights['centeredness']
    sw   = sw if sw > 1e-12 else 1.0
    brs  = (weights['density'] * den + weights['mixture'] * mix + weights['centeredness'] * cen) / sw
    brs  = np.clip(brs, 0.0, 1.0, out=brs)

    return {"path": path, "den": den, "cen": cen, "mix": mix, "brs": brs}

In [32]:
from matplotlib import patches

In [ ]:
# def make_animation_streaming(
#     item_seq,
#     fps=12,
#     save_path=None,
#     data_dir=DATA_DIR,
#     prefetch=6,
#     workers=os.cpu_count() // 2,
#     use_processes=False,
#     weights=None,  # dict: {'density':1.0, 'mixture':1.0, 'centeredness':1.0}
# ):
#     """
#     비인터랙티브 고속 스트리밍 애니메이션.
#     - item_seq: pandas.DataFrame (columns: replay, image_id, gt_0..gt_4)
#     """
#     if weights is None:
#         weights = {'density': 1.0, 'mixture': 1.0, 'centeredness': 1.0}

#     df = item_seq.reset_index(drop=True)
#     T  = len(df)

#     # ---- 병렬 실행기 ----
#     Executor = ProcessPoolExecutor if use_processes else ThreadPoolExecutor
#     pool = Executor(max_workers=workers) if workers and workers > 0 else None

#     # ---- Figure / 5패널 & 진행바 ----
#     fig, axes = plt.subplots(1, 5, figsize=(30, 6), sharex=True, sharey=True, dpi=300)

#     # 첫 프레임
#     p0   = os.path.join(data_dir, "input", "dst", df.loc[0, "replay"], f"{df.loc[0,'image_id']}.npy")
#     arr0 = np.load(p0, mmap_mode="r")

#     over_dyn = _overview_init(axes[0], arr0)
#     axes[0].set_title("Overview")
#     axes[0].set_xlim(0, GRID_W); axes[0].set_ylim(GRID_H, 0); axes[0].set_aspect("equal")

#     den0 = score_density(arr0)
#     cen0 = score_centeredness(arr0)
#     mix0 = score_mixture(arr0, mode="linear")

#     sw   = weights['density'] + weights['mixture'] + weights['centeredness']
#     sw   = sw if sw > 1e-12 else 1.0
#     brs0 = np.clip((weights['density']*den0 + weights['mixture']*mix0 + weights['centeredness']*cen0)/sw, 0.0, 1.0)

#     # 스코어맵 imshow (vmin/vmax 고정: 리스케일 비용 제거)
#     im1 = axes[1].imshow(den0, origin="lower", vmin=0, vmax=1, animated=True); axes[1].set_title("Density")
#     im2 = axes[2].imshow(cen0, origin="lower", vmin=0, vmax=1, animated=True); axes[2].set_title("Centeredness")
#     im3 = axes[3].imshow(mix0, origin="lower", vmin=0, vmax=1, animated=True); axes[3].set_title("Mixture")
#     im4 = axes[4].imshow(brs0, origin="lower", vmin=0, vmax=1, animated=True); axes[4].set_title("BRS")

#     for ax in axes:
#         ax.set_xlim(0, GRID_W); ax.set_ylim(GRID_H, 0); ax.set_aspect("equal")
#         ax.set_xticks([]); ax.set_yticks([])

#     # 진행바(텍스트 없음: 더 빠름)
#     prog_ax = fig.add_axes([0.1, 0.06, 0.8, 0.02])
#     prog_ax.set_xlim(0, 1); prog_ax.set_ylim(0, 1); prog_ax.axis("off")
#     prog_ax.add_artist(patches.Rectangle((0, 0), 1, 1, fill=False, linewidth=1, edgecolor="0.5", transform=prog_ax.transAxes))
#     bar = patches.Rectangle((0, 0), 0.0, 1.0, transform=prog_ax.transAxes, animated=True)
#     prog_ax.add_patch(bar)

#     # GT: rect는 axes[0]만, dot은 전 축
#     max_gts   = 5
#     rects_ax0 = []
#     dots_ax   = {i: [] for i in range(5)}
#     for _ in range(max_gts):
#         r0 = patches.Rectangle((0, 0), 0, 0, linewidth=1, edgecolor="red",
#                                facecolor="none", visible=False, animated=True, zorder=5)
#         axes[0].add_patch(r0); rects_ax0.append(r0)
#         for i in range(5):
#             d = patches.Circle((0, 0), radius=1.0, color="blue",
#                                visible=False, animated=True, zorder=6)
#             axes[i].add_patch(d); dots_ax[i].append(d)

#     # ---- 프리패치 ----
#     futures, results = {}, {}

#     def _submit_until(target_idx):
#         if pool is None: return
#         last_to = min(T - 1, target_idx + prefetch)
#         for i in range(target_idx, last_to + 1):
#             if i in futures or i in results: continue
#             args = (data_dir, df.loc[i, "replay"], df.loc[i, "image_id"], weights)
#             futures[i] = pool.submit(_worker_compute, *args)

#     _submit_until(0)

#     def _get_result(i):
#         if i in results: return results[i]
#         if pool is None:
#             return _worker_compute(data_dir, df.loc[i, "replay"], df.loc[i, "image_id"], weights)
#         res = futures[i].result()
#         results[i] = res
#         old = i - 2
#         if old in results: del results[old]
#         if old in futures: del futures[old]
#         return res

#     # ---- 업데이트 ----
#     def update(t):
#         _submit_until(t + 1)
#         data = _get_result(t)

#         # overview 갱신(cla 금지)
#         arr = np.load(data["path"], mmap_mode="r")
#         _overview_update(axes[0], arr, over_dyn)

#         # 스코어맵 갱신
#         im1.set_data(data["den"])
#         im2.set_data(data["cen"])
#         im3.set_data(data["mix"])
#         im4.set_data(data["brs"])  # 워커에서 미리 계산

#         # GT (좌상단 원점 → Matplotlib 좌표)
#         gts = _extract_gts(df.loc[t])
#         for i in range(max_gts):
#             vis = i < len(gts)
#             rects_ax0[i].set_visible(vis)
#             for ax_i in range(5):
#                 dots_ax[ax_i][i].set_visible(vis)
#             if vis:
#                 x, y, w, h = gts[i]
#                 rects_ax0[i].set_xy((x, y))
#                 rects_ax0[i].set_width(w); rects_ax0[i].set_height(h)
#                 for ax_i in range(5):
#                     dots_ax[ax_i][i].center = (x, y)

#         # 진행바(너비만 갱신)
#         bar.set_width((t + 1) / T)

#         # blit 대상 반환
#         return [*over_dyn, im1, im2, im3, im4, bar,
#                 *rects_ax0, *dots_ax[0], *dots_ax[1], *dots_ax[2], *dots_ax[3], *dots_ax[4]]

#     anim = FuncAnimation(fig, update, frames=T, interval=int(round(1000 / fps)), blit=True)

#     # 저장
#     if save_path:
#         if save_path.lower().endswith(".gif"):
#             anim.save(save_path, writer=PillowWriter(fps=fps))
#         else:
#             from matplotlib.animation import FFMpegWriter
#             anim.save(save_path, writer=FFMpegWriter(fps=fps))
#         print(f"Saved animation to {save_path}")

#     if pool is not None:
#         pool.shutdown(wait=False)

#     return anim, fig, axes

In [34]:
# anim, fig, axes = make_animation_streaming(wide_ground_truth[10000:10050], fps=24, save_path="animation_example_streaming.gif", weights={'density':0.25, 'mixture':1.0, 'centeredness':0.3}, prefetch=8, workers=os.cpu_count() // 2, use_processes=True)

In [35]:
# anim, fig, axes = make_animation_streaming(wide_ground_truth[0:1000], fps=24, save_path="animation_example_streaming.gif", weights={'density':0.25, 'mixture':1.0, 'centeredness':0.3}, prefetch=8, workers=os.cpu_count(), use_processes=True)

In [ ]:
# print(len(wide_ground_truth))

18188


In [ ]:
# interval = 120
# for i in range(0, len(wide_ground_truth), interval):
#     anim, fig, axes = make_animation_streaming(wide_ground_truth[i:i+interval], fps=24, save_path=f"animation_example_streaming_{i}.gif", weights={'density':0.25, 'mixture':1.0, 'centeredness':0.3}, prefetch=8, workers=os.cpu_count(), use_processes=True)

/tmp/ipykernel_1361/1978024771.py:44: RuntimeWarning: invalid value encountered in divide
  return (mix - mix.min()) / (mix.max() - mix.min())
Process ForkProcess-43:
Process ForkProcess-20:
Process ForkProcess-17:
Process ForkProcess-14:
Process ForkProcess-34:
Process ForkProcess-30:
Process ForkProcess-13:
Process ForkProcess-42:
Process ForkProcess-29:
Process ForkProcess-46:
Process ForkProcess-44:
Process ForkProcess-35:
Process ForkProcess-16:
Process ForkProcess-48:
Process ForkProcess-37:
Process ForkProcess-23:
Process ForkProcess-36:
Process ForkProcess-39:
Process ForkProcess-41:
Process ForkProcess-31:
Process ForkProcess-19:
Process ForkProcess-47:
Process ForkProcess-22:
Process ForkProcess-45:
Process ForkProcess-12:
Process ForkProcess-26:
Process ForkProcess-18:
Process ForkProcess-15:
Process ForkProcess-28:
Process ForkProcess-10:
Process ForkProcess-38:
Process ForkProcess-11:
Process ForkProcess-24:
Process ForkProcess-9:
Process ForkProcess-40:
Process ForkProces

KeyboardInterrupt: 